# 14 — Explore Candidate 12-Month Reporting Windows

**Project:** Clinical Microbiology Analytics Lakehouse  
**Source:** Stanford ARMD (public de-identified data)  
**Warehouse:** `workspace.micro_dev`

Use this notebook to inspect the availability of culture-organism observations and evidence of selected resistance phenotypes by **shifted date year**, before choosing an optional 12-month reporting window.

## Purpose and interpretation

The query reads `int_reporting_culture_organism`, which is **not restricted by the optional dbt reporting window**. It therefore remains suitable for exploring candidate shifted years even after a period-specific mart has been built.

- `patient_organism_groups` counts patient–organism groups **within each shifted year and specimen cohort**; the same patient can appear across different shifted years or organisms.
- `groups_with_esce_evidence`, `groups_with_mrsa_evidence`, and `groups_with_vre_evidence` indicate groups with at least one positive phenotype-status observation during that shifted year.
- These are **preselection availability counts**, not first-culture reporting counts or drug-specific categorized AST denominators.
- ARMD's patient-level date shifts mean a shifted year is **not an actual common clinical calendar year** across patients. This analysis can guide a research reporting window but cannot establish institutional annual antibiogram rates.

Run the following `%sql` cell using your Databricks-connected VS Code notebook kernel.

In [1]:
%sql
with distinct_culture_organisms as (
    select distinct
        source_version, cohort_id, patient_key, organism_reporting_group,
        culture_key, cast(culture_time_shifted as date) as shifted_date,
        ast_esce_enterobacterales_status,
        ast_oxacillin_resistance_status,
        ast_vancomycin_resistance_status
    from workspace.micro_dev.int_reporting_culture_organism
    where culture_time_shifted is not null
),
patient_year as (
    select year(shifted_date) as shifted_year, cohort_id,
           organism_reporting_group, patient_key,
           max(case when ast_esce_enterobacterales_status = 'positive'
                     then 1 else 0 end) as any_esce_positive,
           max(case when ast_oxacillin_resistance_status = 'positive'
                     then 1 else 0 end) as any_mrsa_positive,
           max(case when ast_vancomycin_resistance_status = 'positive'
                     then 1 else 0 end) as any_vre_positive
    from distinct_culture_organisms
    group by year(shifted_date), cohort_id, organism_reporting_group, patient_key
)
select shifted_year, cohort_id,
       count(*) as patient_organism_groups,
       sum(any_esce_positive) as groups_with_esce_evidence,
       sum(any_mrsa_positive) as groups_with_mrsa_evidence,
       sum(any_vre_positive) as groups_with_vre_evidence
from patient_year
group by shifted_year, cohort_id
order by shifted_year desc, cohort_id;

,shifted_year,cohort_id,patient_organism_groups,groups_with_esce_evidence,groups_with_mrsa_evidence,groups_with_vre_evidence
0,2024,blood,22,3,1,0
1,2024,respiratory,24,1,1,0
2,2024,urine,183,15,0,0
3,2023,blood,922,86,59,12
4,2023,respiratory,715,21,34,0
5,2023,urine,7528,683,22,16
6,2022,blood,961,74,55,11
7,2022,respiratory,698,20,48,0
8,2022,urine,7273,558,18,26
9,2021,blood,907,72,59,5


## Selecting and validating a reporting window

Review the shifted years for adequate cohort and phenotype evidence. When selecting an exact 12-month period, also assess whether organism–antibiotic pairs have at least **30 categorized AST results** after first-culture selection; the availability counts above cannot determine this on their own.

After configuring the reporting window in dbt, run a full `dbt build`, rerun Notebook 13 against the new marts, and refresh Tableau exports. Clearly label the chosen period as a **shifted-date research window**, not a real calendar-year hospital report.

The SQL above was preserved from `explore_reporting_windows.sql`; this notebook contains no saved warehouse output.